# NB26: International Boundaries — GADM

siege_utilities supports international boundary data through the
**GADM** (Global Administrative Areas Database) model hierarchy.

GADM provides administrative boundaries for every country at up to 5 levels:

| Level | US Equivalent | Example (Mexico) |
|-------|--------------|-------------------|
| 0 | Country | México |
| 1 | State | Jalisco |
| 2 | County | Guadalajara |
| 3 | Township | — |
| 4-5 | Sub-township | — |

The Django models (`GADMCountry`, `GADMAdmin1-5`) store these with temporal
validity, geometry, and parent FK relationships.

The Pydantic schemas work without a database for validation and data pipelines.

In [ ]:
from siege_utilities.geo.schemas.gadm import (
    GADMBoundarySchema,
    GADMCountrySchema,
    GADMAdmin1Schema,
    GADMAdmin2Schema,
)

## 1. Country Level (GADM Level 0)

In [ ]:
# Define countries of interest
countries = [
    GADMCountrySchema(
        feature_id="MEX", name="Mexico", vintage_year=2024,
        gid="MEX", iso3="MEX", source="GADM v4.1",
    ),
    GADMCountrySchema(
        feature_id="CAN", name="Canada", vintage_year=2024,
        gid="CAN", iso3="CAN", source="GADM v4.1",
    ),
    GADMCountrySchema(
        feature_id="GBR", name="United Kingdom", vintage_year=2024,
        gid="GBR", iso3="GBR", source="GADM v4.1",
    ),
    GADMCountrySchema(
        feature_id="DEU", name="Germany", vintage_year=2024,
        gid="DEU", iso3="DEU", source="GADM v4.1",
    ),
]

for c in countries:
    print(f"  {c.iso3}: {c.name} (source: {c.source})")

## 2. Admin Level 1 (States/Provinces)

In [ ]:
# Mexican states (selected)
mexican_states = [
    GADMAdmin1Schema(
        feature_id=f"MEX.{i}", name=name, vintage_year=2024,
        gid=f"MEX.{i}_1", gid_0_string="MEX",
        type_1="Estado", engtype_1="State", source="GADM v4.1",
    )
    for i, name in enumerate([
        "Jalisco", "Ciudad de México", "Nuevo León",
        "Guanajuato", "Puebla", "Chihuahua",
    ], start=1)
]

print(f"Mexican states ({len(mexican_states)} shown):")
for s in mexican_states:
    print(f"  {s.gid}: {s.name} ({s.engtype_1})")

# Canadian provinces
canadian_provinces = [
    GADMAdmin1Schema(
        feature_id=f"CAN.{i}", name=name, vintage_year=2024,
        gid=f"CAN.{i}_1", gid_0_string="CAN",
        type_1="Province" if prov else "Territory",
        engtype_1="Province" if prov else "Territory",
        source="GADM v4.1",
    )
    for i, (name, prov) in enumerate([
        ("Ontario", True), ("Quebec", True), ("British Columbia", True),
        ("Alberta", True), ("Yukon", False), ("Northwest Territories", False),
    ], start=1)
]

print(f"\nCanadian provinces/territories ({len(canadian_provinces)} shown):")
for p in canadian_provinces:
    print(f"  {p.gid}: {p.name} ({p.engtype_1})")

## 3. Admin Level 2 (Counties/Municipalities)

In [ ]:
# Municipalities within Jalisco
jalisco_municipios = [
    GADMAdmin2Schema(
        feature_id=f"MEX.1.{i}", name=name, vintage_year=2024,
        gid=f"MEX.1.{i}_1", gid_1_string="MEX.1_1",
        type_2="Municipio", engtype_2="Municipality", source="GADM v4.1",
    )
    for i, name in enumerate([
        "Guadalajara", "Zapopan", "Tlaquepaque",
        "Tonalá", "Puerto Vallarta",
    ], start=1)
]

print(f"Jalisco municipalities ({len(jalisco_municipios)} shown):")
for m in jalisco_municipios:
    print(f"  {m.gid}: {m.name} ({m.engtype_2})")

## 4. The GADM Model Hierarchy

In production (with PostGIS), the Django models form a hierarchy:

```
GADMCountry (iso3="MEX")
├── GADMAdmin1 (gid="MEX.1_1", name="Jalisco")
│   ├── GADMAdmin2 (gid="MEX.1.1_1", name="Guadalajara")
│   │   ├── GADMAdmin3 (if available)
│   │   │   └── GADMAdmin4, GADMAdmin5 (finest granularity)
│   ├── GADMAdmin2 (gid="MEX.1.2_1", name="Zapopan")
│   └── ...
├── GADMAdmin1 (gid="MEX.2_1", name="Ciudad de México")
└── ...
```

Key features:
- All models inherit from `TemporalBoundary` (MultiPolygon geometry, vintage_year)
- FK relationships link child → parent
- `populate_parent_relationships()` resolves string GIDs to FK objects after bulk loading
- Source is always "GADM" with version tag

## 5. Country Code Utilities

siege_utilities includes a full country code mapping (270+ entries).

In [ ]:
from siege_utilities.geo.geocoding import get_country_name, get_country_code, list_countries

# Look up by code
for code in ["mx", "ca", "gb", "de", "jp", "br"]:
    print(f"  {code.upper()}: {get_country_name(code)}")

# Reverse lookup
print(f"\n  Mexico → {get_country_code('Mexico')}")
print(f"  Canada → {get_country_code('Canada')}")

all_countries = list_countries()
print(f"\nTotal countries in registry: {len(all_countries)}")

## Summary

| Model | Level | Example |
|-------|-------|---------|
| `GADMCountry` | 0 | Mexico, Canada, UK |
| `GADMAdmin1` | 1 | Jalisco, Ontario, Bavaria |
| `GADMAdmin2` | 2 | Guadalajara, Toronto, Munich |
| `GADMAdmin3` | 3 | Sub-municipal |
| `GADMAdmin4-5` | 4-5 | Finest granularity |

**Data source:** [GADM.org](https://gadm.org) — free for academic and non-commercial use.

**Loading:** Use the `GADMProvider` service (or direct ORM bulk creation)
to populate boundaries from GADM shapefiles/GeoJSON downloads.